# Stronger detectors for HALT

Two baselines the paper needs and the analysis container cannot run: a
fine-tuned encoder, and an LLM-as-detector.

**Why this matters.** The paper's central claim is that hallucination detection
does not transfer across source LLMs. Every detector reported so far is linear.
A reviewer will ask whether the collapse is a property of the task or an
artifact of weak detectors, and right now there is no answer. This notebook
produces one.

Write the result whichever way it comes out. If these detectors collapse too,
the claim stands and gets stronger. If they do not, Section 4.3 gets reframed
as *which* detectors read evidence rather than style, which is also a finding.

**Runtime:** encoder about 2.5 h on an A100. LLM detector depends on how much
you sample; start small.

**Set the runtime to GPU before running Part 1:** Runtime → Change runtime type
→ A100 (or any GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Direct path first. The fallback is depth-limited so it cannot walk the whole
# drive over the network mount.
CANDIDATES = [
    '/content/drive/Shareddrives/RESEARCH/2026/PROMPTS/RESULTS',
    '/content/drive/Shareddrives/RESEARCH/Research/2026/PROMPTS/RESULTS',
]

def ok(p):
    return os.path.isdir(os.path.join(p, 'EVALUATION-RESULT'))

RESULTS_ROOT = next((p for p in CANDIDATES if ok(p)), None)

if RESULTS_ROOT is None:
    for base in ['/content/drive/Shareddrives', '/content/drive/MyDrive']:
        if not os.path.isdir(base):
            continue
        depth0 = base.rstrip('/').count('/')
        for root, dirs, _ in os.walk(base):
            if root.count('/') - depth0 >= 4:
                dirs[:] = []
                continue
            if 'RESULTS' in dirs and ok(os.path.join(root, 'RESULTS')):
                RESULTS_ROOT = os.path.join(root, 'RESULTS'); break
        if RESULTS_ROOT:
            break

if RESULTS_ROOT is None:
    raise SystemExit('Not found. Set RESULTS_ROOT by hand.')

OUT = f'{RESULTS_ROOT}/ANALYSIS-OUTPUT-RESULT/iclr_benchmark'
os.environ['HALLUBENCH_OUT'] = OUT
LABELS = f'{OUT}/benchmark_labels.csv.gz'

print('OUT    =', OUT)
print('labels =', LABELS, '->', 'found' if os.path.exists(LABELS) else 'MISSING')

In [ ]:
import pandas as pd
df = pd.read_csv(LABELS)
print(df.shape)
print('splits:', [c for c in df.columns if c.startswith('split_')])
print('source LLMs:', sorted(df.model.unique()))
print('reports with text:', int(df.model_output.notna().sum()))

---
# Part 1 — Fine-tuned encoder

One shared encoder, seven binary heads (H1–H6 plus ANY), fed
`[report] [SEP] [reference]`. Class weights match the balanced logistic
baselines so the comparison is like-for-like.

Trains five times: the standard split, each of the three leave-one-source-out
folds, and held-out-strategy. Writes `encoder_results.csv` in the same schema
as `baseline_results.csv`, so the existing report and figure code reads it
without changes.

**Do a cheap pass first.** Run with `roberta-base` and `--epochs 1` to confirm
the loop works end to end before committing to the full run.

In [ ]:
!pip -q install "transformers>=4.48" accelerate
import torch, transformers
print('transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — switch runtime to GPU')

In [ ]:
%%writefile /content/06_encoder_detector.py
#!/usr/bin/env python3
"""
Step 6 - fine-tuned encoder detector. RUN THIS ON A GPU (Colab A100 is enough).

Addresses the strongest objection to the paper: that the transfer collapse is
an artifact of linear detectors rather than a property of the task. Fine-tunes
one encoder per split with six per-type heads plus an ANY head, on
[report] [SEP] [reference], and evaluates on the standard, leave-one-source-out
(all three folds), and held-out-strategy splits.

Requires network access to download model weights, so it cannot run in the
offline analysis container.

    pip install "transformers>=4.48" torch scikit-learn accelerate
    python3 06_encoder_detector.py --model answerdotai/ModernBERT-base

CONTEXT LENGTH MATTERS HERE. Reports average ~1,500 tokens. The judge that
produced the labels saw 3,000 characters of report (~750 tokens) plus 1,500 of
reference (~375), so --max_len 1280 gives the detector exactly the judge's
view. At 512 the detector sees less than the judge did and the comparison is
unfair to it. ModernBERT handles 8,192 tokens, so 1,280 costs nothing.

Writes: encoder_results.csv  (same schema as baseline_results.csv, so the
        existing report and figure code consumes it unchanged)

Runtime guide, A100, ModernBERT-base, max_len 1280, bs 8, 2 epochs:
    standard split          ~60 min
    3 LOSO folds            ~150 min
    held-out strategy       ~60 min
Roughly 4.5 h in total. For the validation pass use
    --model roberta-base --max_len 512 --bs 16 --epochs 1
which takes about 20 minutes and only checks that the loop runs.
"""
import argparse, os
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

ap = argparse.ArgumentParser()
ap.add_argument('--model', default='answerdotai/ModernBERT-base')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/encoder_results.csv')
ap.add_argument('--max_len', type=int, default=1280,
                help='1280 matches what the judge saw; raise it to test whether '
                     'the detector benefits from more than the judge had')
ap.add_argument('--epochs', type=int, default=2)
ap.add_argument('--bs', type=int, default=8,
                help='lower than usual because of the long context')
ap.add_argument('--lr', type=float, default=2e-5)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev == 'cpu':
    print('WARNING: no GPU visible. This will take many hours.')

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')
tok = AutoTokenizer.from_pretrained(args.model)
_cap = getattr(tok, 'model_max_length', 512)
if _cap and _cap < args.max_len and _cap < 100000:
    print(f'WARNING: {args.model} caps at {_cap} tokens but --max_len is '
          f'{args.max_len}. Reports will be cut below what the judge saw. '
          f'Use a long-context model (ModernBERT, Longformer) or lower '
          f'--max_len and say so in the paper.')


class Reports(Dataset):
    """Report and its reference annotation as a sentence pair."""
    def __init__(self, frame):
        self.a = frame.model_output.tolist()
        self.b = frame.ground_truth.tolist()
        self.y = frame[TARGETS].to_numpy(dtype='float32')

    def __len__(self):
        return len(self.a)

    def __getitem__(self, i):
        enc = tok(self.a[i], self.b[i], truncation=True, max_length=args.max_len,
                  padding='max_length', return_tensors='pt')
        return ({k: v.squeeze(0) for k, v in enc.items()},
                torch.tensor(self.y[i]))


class MultiHead(torch.nn.Module):
    """One shared encoder, seven independent binary heads."""
    def __init__(self, name, n=len(TARGETS)):
        super().__init__()
        # force fp32: some checkpoints (DeBERTa-v3) declare a fp16 dtype in
        # their config, which makes GradScaler refuse to unscale gradients
        self.enc = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        d = self.enc.config.hidden_size
        self.drop = torch.nn.Dropout(0.1)
        self.heads = torch.nn.Linear(d, n)

    def forward(self, **kw):
        h = self.enc(**kw).last_hidden_state[:, 0]     # [CLS]
        return self.heads(self.drop(h))


def run(train_mask, test_mask, tag):
    tr, te = df[train_mask].reset_index(drop=True), df[test_mask].reset_index(drop=True)
    print(f'\n== {tag}: train {len(tr):,}  test {len(te):,}', flush=True)

    model = MultiHead(args.model).to(dev)
    dl_tr = DataLoader(Reports(tr), batch_size=args.bs, shuffle=True, num_workers=2)
    dl_te = DataLoader(Reports(te), batch_size=args.bs * 2, num_workers=2)

    # class weights per head, matching the balanced logistic baselines
    pos = tr[TARGETS].mean().to_numpy()
    w = torch.tensor(((1 - pos) / np.clip(pos, 1e-6, None)).astype('float32')).to(dev)
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr)
    steps = len(dl_tr) * args.epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)

    # bf16 where the GPU supports it (A100 and newer): same dynamic range as
    # fp32, so no loss scaling is needed and GradScaler is skipped entirely.
    use_bf16 = dev == 'cuda' and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda' and not use_bf16))
    print(f'   precision: {"bf16" if use_bf16 else ("fp16+scaler" if dev=="cuda" else "fp32")}',
          flush=True)

    model.train()
    for ep in range(args.epochs):
        for i, (x, y) in enumerate(dl_tr):
            x = {k: v.to(dev) for k, v in x.items()}; y = y.to(dev)
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                loss = lossf(model(**x), y)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            sch.step()
            if i % 100 == 0:
                print(f'   ep{ep} step {i}/{len(dl_tr)} loss {loss.item():.4f}', flush=True)

    model.eval(); P = []
    with torch.no_grad():
        for x, _ in dl_te:
            x = {k: v.to(dev) for k, v in x.items()}
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                P.append(torch.sigmoid(model(**x)).float().cpu().numpy())
    P = np.vstack(P)

    rows = []
    for j, t in enumerate(TARGETS):
        y = te[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'encoder', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, P[:, j]),
                     'f1': f1_score(y, (P[:, j] >= 0.5).astype(int)),
                     'pos_rate_test': float(y.mean())})
    del model; torch.cuda.empty_cache()
    return rows


out = []
out += run(df.split_random == 'train', df.split_random == 'test', 'random')
for held in ['Claude', 'GPT', 'Gemini']:
    out += run(df.model != held, df.model == held, f'heldout_model_{held}')
out += run(df.split_heldout_technique == 'train',
           df.split_heldout_technique == 'test', 'heldout_technique')

res = pd.DataFrame(out)
res.to_csv(args.out, index=False)
print('\n' + res.pivot_table(index='split', columns='target',
                             values='auc')[TARGETS].round(3).to_markdown())
print('\nwrote ->', args.out)


### Cheap validation pass

RoBERTa at 512 tokens, one epoch, about 20 minutes. This only checks that the
training loop runs and the table populates. Do not read the numbers as results:
at 512 tokens the detector sees less than the judge did.

In [ ]:
!python3 /content/06_encoder_detector.py \\
    --model roberta-base --max_len 512 --bs 16 --epochs 1 \\
    --labels "$LABELS" --out "$OUT/encoder_results_smoke.csv" 

### Full run

ModernBERT-base at `max_len 1280`. That figure is chosen deliberately: the
judge that produced the labels saw 3,000 characters of report (~750 tokens)
plus 1,500 of reference (~375), so 1,280 gives the detector exactly the judge's
view. Any less and the comparison is unfair to the detector.

About 4.5 hours on an A100 across all five conditions. Colab disconnects on
idle, so keep the tab active.

Optional afterwards: rerun one fold at `--max_len 2048`. If the detector
improves when it sees more than the judge did, that is evidence the labels
themselves are truncation-limited, which connects to the analysis in Section
4.5.

In [ ]:
!python3 /content/06_encoder_detector.py \\
    --model answerdotai/ModernBERT-base --max_len 1280 --bs 8 --epochs 2 \\
    --labels "$LABELS" --out "$OUT/encoder_results.csv" 

In [ ]:
import os, pandas as pd
T = ['H1','H2','H3','H4','H5','H6','any_hallucination']; H = T[:6]

path = f'{OUT}/encoder_results.csv'
if not os.path.exists(path):
    path = f'{OUT}/encoder_results_smoke.csv'
    print('full run not found; reading the smoke run instead:', path, '\n')
e = pd.read_csv(path)

piv = e.pivot_table(index='split', columns='target', values='auc')[T]
piv['macro'] = piv[H].mean(axis=1)
print(piv.round(3).to_markdown())

loso = [i for i in piv.index if i.startswith('heldout_model')]
if loso and 'random' in piv.index:
    print()
    print(f"random macro           {piv.loc['random','macro']:.3f}")
    print(f"leave-one-source-out   {piv.loc[loso,'macro'].mean():.3f}")
    print(f"random ANY             {piv.loc['random','any_hallucination']:.3f}")
    print(f"LOSO ANY               {piv.loc[loso,'any_hallucination'].mean():.3f}")
    print()
    print('linear baselines: tfidf macro 0.846 -> 0.723, ANY 0.820 -> 0.566')

---
# Part 2 — LLM-as-detector

Zero-shot. A fourth model, **not** one of the three that wrote the corpus —
prompting one of the three would reintroduce the self-preference bias the panel
design exists to exclude, and the result would be uninterpretable.

Inference only, so there is no training split. The "held-out source" condition
is simply the reports from that source LLM.

**Two cautions.**

Cost. The full corpus is 19,361 reports; scoring every condition is roughly
25,000 calls. `--n-per-fold` subsamples each test set stratified by crime type.
Start at 300 to validate the prompt and the JSON parsing, then raise to 1500,
which is enough for AUC at the precision the paper reports.

Comparability. This detector emits hard 0/1 labels, so its AUC is computed on
binary predictions and is **not** directly comparable to the probabilistic
baselines. Report F1 beside it and say so in the table caption.

In [ ]:
# Install the SDK for whichever provider you use, then set the key.
!pip -q install openai
import os, getpass
os.environ['DETECTOR_API_KEY'] = getpass.getpass('API key: ')

In [ ]:
%%writefile /content/07_llm_detector.py
#!/usr/bin/env python3
"""
Step 7 - LLM-as-detector baseline. RUN THIS WHERE YOU HAVE API ACCESS.

Prompts a fourth model, NOT one of the three source LLMs, zero-shot with the
report and its reference annotation, and asks for the six binary labels. Using
a non-source model matters: prompting one of the three would reintroduce the
self-preference bias the panel design was built to exclude, and would make the
result uninterpretable.

Zero-shot, so there is no training split. The same test sets as the other
detectors are used, and because this is inference-only the "held-out source"
condition is simply the reports from that source LLM.

    pip install openai            # or anthropic / google-generativeai
    export DETECTOR_API_KEY=...
    python3 07_llm_detector.py --model <a-model-not-in-the-corpus> --n-per-fold 1500

Writes: llm_detector_results.csv (schema matches baseline_results.csv)
        llm_detector_raw.csv     (per-report verdicts, for auditing)

COST WARNING. The full corpus is 19,361 reports. Scoring every test set would
be roughly 25,000 calls. --n-per-fold subsamples each test set, stratified by
crime type, which is enough for AUC at the precision this paper reports.
Start with --n-per-fold 300 to sanity-check the prompt and the parsing before
spending real money.
"""
import argparse, json, os, re, time
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import roc_auc_score, f1_score

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

RUBRIC = """You are auditing a forensic video report against an expert reference
annotation of the same video. Decide, for each category, whether the report
contains that error.

H1 scene fabrication: describes a setting or event with no source in the footage
H2 crime misclassification: reports the act as a different category of crime
H3 crime missed: fails to report the criminal act under investigation
H4 severity minimization: understates the gravity of the act
H5 entity fabrication: introduces objects or details not present
H6 phantom actors: introduces people who do not appear

REFERENCE ANNOTATION:
{ref}

REPORT:
{rep}

Answer with JSON only, no prose:
{{"H1":0or1,"H2":0or1,"H3":0or1,"H4":0or1,"H5":0or1,"H6":0or1}}"""

ap = argparse.ArgumentParser()
ap.add_argument('--model', required=True,
                help='a model NOT among the three source LLMs')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/llm_detector_results.csv')
ap.add_argument('--raw', default=os.environ.get('HALLUBENCH_OUT', '.') + '/llm_detector_raw.csv')
ap.add_argument('--n-per-fold', type=int, default=1500)
ap.add_argument('--workers', type=int, default=8)
ap.add_argument('--max-ref', type=int, default=1500)
ap.add_argument('--max-rep', type=int, default=3000)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()


def call(prompt):
    """Replace the body with your provider's SDK. Must return the raw string."""
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['DETECTOR_API_KEY'])
    r = client.chat.completions.create(
        model=args.model, temperature=0,
        messages=[{'role': 'user', 'content': prompt}])
    return r.choices[0].message.content


def parse(txt):
    """Pull the six labels out of the reply. Returns None if unparseable."""
    m = re.search(r'\{[^{}]*\}', txt or '', re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except Exception:
        return None
    if not all(h in d for h in H):
        return None
    return {h: int(bool(int(d[h]))) for h in H}


def score(frame):
    """Score one test set. Retries twice on failure, then records a miss."""
    def one(i, row):
        p = RUBRIC.format(ref=str(row.ground_truth)[:args.max_ref],
                          rep=str(row.model_output)[:args.max_rep])
        for attempt in range(3):
            try:
                v = parse(call(p))
                if v is not None:
                    return i, v
            except Exception:
                time.sleep(2 ** attempt)
        return i, None

    res = {}
    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(one, i, r) for i, r in frame.iterrows()]
        for k, f in enumerate(as_completed(futs)):
            i, v = f.result(); res[i] = v
            if k % 100 == 0:
                print(f'   {k}/{len(frame)}', flush=True)
    return res


df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')
rng = np.random.default_rng(args.seed)


def subsample(frame):
    if len(frame) <= args.n_per_fold:
        return frame
    per = max(1, args.n_per_fold // frame.crime_type.nunique())
    return (frame.groupby('crime_type', group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), per), random_state=args.seed)))


conditions = {'random': df[df.split_random == 'test'],
              'heldout_technique': df[df.split_heldout_technique == 'test']}
for held in ['Claude', 'GPT', 'Gemini']:
    conditions[f'heldout_model_{held}'] = df[df.model == held]

rows, raw = [], []
for tag, frame in conditions.items():
    sub = subsample(frame).copy()
    print(f'\n== {tag}: scoring {len(sub):,} reports', flush=True)
    verdicts = score(sub)
    ok = [i for i, v in verdicts.items() if v is not None]
    print(f'   parsed {len(ok)}/{len(sub)}')
    sub = sub.loc[ok]
    P = pd.DataFrame([verdicts[i] for i in ok], index=ok)
    P['any_hallucination'] = P[H].max(axis=1)
    for h in TARGETS:
        raw.append(pd.DataFrame({'split': tag, 'target': h,
                                 'video': sub.video.values, 'model': sub.model.values,
                                 'technique': sub.technique.values,
                                 'pred': P[h].values, 'gold': sub[h].values}))
    for t in TARGETS:
        y, p = sub[t].to_numpy(), P[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'llm_detector', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, p), 'f1': f1_score(y, p),
                     'pos_rate_test': float(y.mean()), 'n': len(y)})

res = pd.DataFrame(rows); res.to_csv(args.out, index=False)
pd.concat(raw).to_csv(args.raw, index=False)
print('\n' + res.pivot_table(index='split', columns='target',
                             values='auc')[TARGETS].round(3).to_markdown())
print('\nNOTE: this detector emits hard 0/1 labels, so its AUC is computed on '
      'binary predictions and is not directly comparable to the probabilistic '
      'baselines. Report F1 alongside it, and say so in the caption.')
print('wrote ->', args.out)


### Validation pass, 300 reports per condition

Check the `parsed n/n` line in the output. If parsing is failing, fix
`parse()` or the prompt before scaling up — that is the whole point of this
pass.

Replace `MODEL` with a model that is **not** Claude, GPT, or Gemini.

In [ ]:
MODEL = 'REPLACE-ME'   # must not be one of the three source LLMs

!python3 /content/07_llm_detector.py \
    --model "$MODEL" --n-per-fold 300 \
    --labels "$LABELS" \
    --out "$OUT/llm_detector_results_smoke.csv" \
    --raw "$OUT/llm_detector_raw_smoke.csv" 

### Full run

ModernBERT-base at `max_len 1280`. That figure is chosen deliberately: the
judge that produced the labels saw 3,000 characters of report (~750 tokens)
plus 1,500 of reference (~375), so 1,280 gives the detector exactly the judge's
view. Any less and the comparison is unfair to the detector.

About 4.5 hours on an A100 across all five conditions. Colab disconnects on
idle, so keep the tab active.

Optional afterwards: rerun one fold at `--max_len 2048`. If the detector
improves when it sees more than the judge did, that is evidence the labels
themselves are truncation-limited, which connects to the analysis in Section
4.5.

In [ ]:
!python3 /content/07_llm_detector.py \
    --model "$MODEL" --n-per-fold 1500 --workers 8 \
    --labels "$LABELS" \
    --out "$OUT/llm_detector_results.csv" \
    --raw "$OUT/llm_detector_raw.csv" 

---
# Part 3 — Combined view

What goes into the paper: whether the two stronger detectors collapse under
source-LLM shift the way the linear ones do.

In [ ]:
import pandas as pd
T = ['H1','H2','H3','H4','H5','H6','any_hallucination']; H = T[:6]
AX = {'fabrication':['H1','H5','H6'], 'omission':['H3'], 'distortion':['H2','H4']}

frames = []
for f, name in [('baseline_results.csv','linear'),
                ('encoder_results.csv','encoder'),
                ('llm_detector_results.csv','llm_detector')]:
    try:
        frames.append(pd.read_csv(f'{OUT}/{f}'))
    except FileNotFoundError:
        print('missing (skipped):', f)
allr = pd.concat(frames, ignore_index=True)

allr['condition'] = allr.split.str.replace(r'heldout_model_.*', 'heldout_model',
                                           regex=True)
piv = allr.pivot_table(index=['features','condition'], columns='target',
                       values='auc')[T]
for a, cs in AX.items():
    piv[a] = piv[cs].mean(axis=1)
piv['macro'] = piv[H].mean(axis=1)
out = piv.round(3)
out.to_csv(f'{OUT}/all_detectors_summary.csv')
print(out.to_markdown())

### Send back

Paste the output of the two cells above, and share these files:

- `encoder_results.csv`
- `llm_detector_results.csv`
- `all_detectors_summary.csv`

They become rows in Tables 3 and 4 and bars in Figure 3, and Section 4.3 gets
written to match whichever way the numbers fall.